In [ ]:

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer
from scipy import stats
import os

def load_data(file_path):
    return pd.read_csv(file_path)

def parse_height(height_str):
    if pd.isna(height_str):
        return None, None
    try:
        feet, inches = height_str.split("'")
        feet = int(feet.strip())
        inches = int(inches.replace('"', '').strip())
    except ValueError:
        return None, None
    return feet, inches

def remove_outliers(df, features, method='zscore', threshold=3):
    if method == 'zscore':
        z_scores = np.abs(stats.zscore(df[features], nan_policy='omit'))
        mask = (z_scores < threshold) | np.isnan(z_scores)
        df = df[mask.all(axis=1)].copy()
    elif method == 'iqr':
        Q1 = df[features].quantile(0.25)
        Q3 = df[features].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        cond = ~((df[features] < lower) | (df[features] > upper)).any(axis=1)
        df = df[cond].copy()
    return df

def preprocess_data(df):
    if 'Gender' in df.columns:
        df = pd.get_dummies(df, columns=['Gender'])
    if 'Cup Size' in df.columns:
        df['Cup Size'] = df['Cup Size'].fillna('None')
        df = pd.get_dummies(df, columns=['Cup Size'])
    features = ['Height', 'Weight', 'Bust/Chest', 'Waist', 'Hips', 'Body Shape Index']
    present = [f for f in features if f in df.columns]
    if len(present) == 0:
        raise ValueError('No required numeric features found.')
    imputer = SimpleImputer(strategy='mean')
    df[present] = imputer.fit_transform(df[present])
    df = remove_outliers(df, present)
    return df, present

def cluster_data(df, features, n_clusters=5):
    scaler = StandardScaler()
    scaled = scaler.fit_transform(df[features])
    scaled_df = pd.DataFrame(scaled, columns=features, index=df.index)
    important_features = ['Body Shape Index']
    for feature in important_features:
        if feature in scaled_df.columns:
            scaled_df[feature] = scaled_df[feature] * 3.0
    kmeans = KMeans(n_clusters=n_clusters, n_init=10, random_state=42)
    df['Cluster'] = kmeans.fit_predict(scaled_df.values)
    return df, kmeans, scaler, important_features

def _unweight_centers(centers, features, important_features, weight=3.0):
    centers_unweighted = centers.copy()
    for feature in important_features:
        if feature in features:
            idx = features.index(feature)
            centers_unweighted[:, idx] = centers_unweighted[:, idx] / weight
    return centers_unweighted

def generate_size_chart(df, features, kmeans, scaler, important_features):
    centers_scaled_weighted = kmeans.cluster_centers_
    centers_scaled = _unweight_centers(centers_scaled_weighted, features, important_features, 3.0)
    centers = scaler.inverse_transform(centers_scaled)
    if 'Height' in features:
        sort_idx = np.argsort(centers[:, features.index('Height')])
        centers = centers[sort_idx]
    size_chart = pd.DataFrame(centers, columns=features)
    size_labels = [str(i+1) for i in range(len(size_chart))]
    size_chart['Size'] = size_labels
    centers_by_cluster = scaler.inverse_transform(_unweight_centers(kmeans.cluster_centers_, features, important_features, 3.0))
    def calculate_confidence(cluster):
        data_c = df[df['Cluster'] == cluster]
        if len(data_c) == 0:
            return 0.0
        means = data_c[features].mean()
        stds = data_c[features].std(ddof=0).replace(0, np.nan)
        cv_inv = 1 / (1 + (stds / means).mean(skipna=True))
        cluster_size = len(data_c) / len(df)
        if 'Body Shape Index' in features:
            bsi_std = data_c['Body Shape Index'].std(ddof=0)
            bsi_mean = data_c['Body Shape Index'].mean()
            bsi_consistency = 1 / (1 + (bsi_std / bsi_mean if bsi_mean not in [0, np.nan] else 0))
        else:
            bsi_consistency = 1.0
        center = centers_by_cluster[cluster]
        dists = np.linalg.norm(data_c[features].values - center, axis=1)
        denom = np.linalg.norm(center) if np.linalg.norm(center) != 0 else 1.0
        closeness = 1 / (1 + dists.mean() / denom) if len(dists) > 0 else 0.0
        scores = np.array([cv_inv*2, cluster_size*2, bsi_consistency*3, closeness*3], dtype=float)
        score = np.nanmean(scores)
        return float(max(0.0, min(1.0, score/2)))
    confidences = []
    for i in range(kmeans.n_clusters):
        confidences.append(calculate_confidence(i))
    if 'Height' in features:
        original_indices = np.argsort(centers_by_cluster[:, features.index('Height')])
        confidences_sorted = [confidences[i] for i in original_indices]
    else:
        confidences_sorted = confidences
    size_chart['Confidence'] = confidences_sorted
    return size_chart

def main():
    file_path = '/content/body_m.csv'
    if not os.path.exists(file_path):
        print('File not found:', file_path)
        return
    df = load_data(file_path)
    print('Available columns:', df.columns.tolist())
    df, features = preprocess_data(df)
    df, kmeans, scaler, important_features = cluster_data(df, features)
    size_chart = generate_size_chart(df, features, kmeans, scaler, important_features)
    print('Generated Size Chart:')
    print(size_chart)
    out_csv = 'generated_size_chart.csv'
    size_chart.to_csv(out_csv, index=False)
    print(\"Size chart saved to\", out_csv)

if __name__ == '__main__':
    main()
